# Module 3: Consumer Groups

本章目标：
- 理解 Consumer Group 如何在多个 Consumer 间分配 Partition
- 观察同一 Group 内的 Consumer 不重复消费同一条消息
- 理解 Rebalance 的触发条件与影响
- 对比不同 Group 消费同一 Topic 的行为

---

## 前置条件

- 集群运行中：`docker compose up -d`
- 已安装依赖：`uv sync`

## 核心概念

```
Topic: events (3 partitions)

Consumer Group A              Consumer Group B
┌──────────────────────┐     ┌──────────────────────┐
│ Consumer-A1 ← P0    │     │ Consumer-B1 ← P0,P1,P2│  (独立消费全量)
│ Consumer-A2 ← P1    │     └──────────────────────┘
│ Consumer-A3 ← P2    │
└──────────────────────┘
      (负载均衡)
```

**关键规则：**
- 同一 Group 内，每个 Partition 只分配给一个 Consumer
- Consumer 数量 > Partition 数量时，多余的 Consumer 空闲
- 不同 Group 完全独立，互不影响，各自从头消费

In [ ]:
import asyncio
from aiokafka import AIOKafkaProducer, AIOKafkaConsumer
from aiokafka.admin import AIOKafkaAdminClient, NewTopic

BROKERS = "localhost:19094,localhost:29094,localhost:39094"
TOPIC = "consumer-groups-demo"
NUM_PARTITIONS = 3

## 1. 准备测试数据

In [ ]:
async def setup(brokers, topic, num_partitions):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        if topic in existing:
            await admin.delete_topics([topic])
            await asyncio.sleep(2)
        await admin.create_topics([
            NewTopic(name=topic, num_partitions=num_partitions, replication_factor=3)
        ])
        await asyncio.sleep(1)
        print(f"✓ Topic '{topic}' ready")
    finally:
        await admin.close()

    producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await producer.start()
    try:
        for i in range(12):
            key = f"key-{i % num_partitions}"  # 确保每分区有数据
            await producer.send_and_wait(topic, key=key.encode(), value=f"event-{i:02d}".encode())
        print(f"✓ Sent 12 messages to {num_partitions} partitions")
    finally:
        await producer.stop()

await setup(BROKERS, TOPIC, NUM_PARTITIONS)

## 2. 单 Group 多 Consumer：观察分区分配

In [ ]:
async def consumer_worker(brokers, topic, group, consumer_id, result_queue):
    """一个 Consumer 实例，消费分配到的分区，将结果放入队列。"""
    consumer = AIOKafkaConsumer(
        topic,
        bootstrap_servers=brokers,
        group_id=group,
        auto_offset_reset="earliest",
    )
    await consumer.start()
    received = []
    try:
        await asyncio.sleep(1.5)  # 等待 Rebalance 完成
        assigned_partitions = {tp.partition for tp in consumer.assignment()}
        # 用 getmany 读取已有消息，idle 超过 1s 则认为读完
        while True:
            results = await consumer.getmany(timeout_ms=1000, max_records=50)
            if not results:
                break
            for tp_msgs in results.values():
                received.extend(tp_msgs)
    finally:
        await consumer.stop()
    await result_queue.put((consumer_id, assigned_partitions, received))

async def demo_consumer_group(brokers, topic, num_consumers=3):
    group = f"demo-group-{num_consumers}c"
    queue = asyncio.Queue()
    workers = [
        consumer_worker(brokers, topic, group, f"C{i}", queue)
        for i in range(num_consumers)
    ]
    await asyncio.gather(*workers)

    results = []
    while not queue.empty():
        results.append(await queue.get())

    print(f"\n=== Group with {num_consumers} consumers ===")
    for cid, partitions, msgs in sorted(results):
        print(f"  {cid}: assigned partitions={sorted(partitions)}, messages={len(msgs)}")

# 3 Consumers, 3 Partitions → 每个 Consumer 分到 1 个 Partition
await demo_consumer_group(BROKERS, TOPIC, num_consumers=3)

In [ ]:
# 1 Consumer, 3 Partitions → 一个 Consumer 消费全部分区
await demo_consumer_group(BROKERS, TOPIC, num_consumers=1)

In [ ]:
# 4 Consumers, 3 Partitions → 有 1 个 Consumer 空闲（无分区分配）
await demo_consumer_group(BROKERS, TOPIC, num_consumers=4)

## 3. 不同 Group 独立消费

两个不同的 Consumer Group 消费同一 Topic，互不影响，各自维护独立的 offset。

In [ ]:
async def consume_as_group(brokers, topic, group_id, label):
    consumer = AIOKafkaConsumer(
        topic, bootstrap_servers=brokers, group_id=group_id,
        auto_offset_reset="earliest",
    )
    await consumer.start()
    count = 0
    try:
        while True:
            results = await consumer.getmany(timeout_ms=1500, max_records=50)
            if not results:
                break
            for tp_msgs in results.values():
                count += len(tp_msgs)
    finally:
        await consumer.stop()
    print(f"  {label} (group={group_id!r}): consumed {count} messages")

print("=== Two independent groups consuming the same topic ===")
await asyncio.gather(
    consume_as_group(BROKERS, TOPIC, "analytics-group", "Analytics"),
    consume_as_group(BROKERS, TOPIC, "audit-group",     "Audit    "),
)
print("Both groups received all 12 messages independently.")

## 4. Offset 提交与断点续传

In [ ]:
PERSISTENT_GROUP = "persistent-group"

async def consume_first_half(brokers, topic, group, stop_after=6):
    consumer = AIOKafkaConsumer(
        topic, bootstrap_servers=brokers, group_id=group,
        auto_offset_reset="earliest",
    )
    await consumer.start()
    try:
        count = 0
        async for msg in consumer:
            print(f"  [Run 1] partition={msg.partition}, offset={msg.offset}: {msg.value.decode()}")
            count += 1
            if count >= stop_after:
                print(f"  [Run 1] Stopping after {stop_after} messages")
                break
    finally:
        await consumer.stop()  # auto-commit on stop

async def consume_remainder(brokers, topic, group):
    consumer = AIOKafkaConsumer(
        topic, bootstrap_servers=brokers, group_id=group,
        auto_offset_reset="earliest",
    )
    await consumer.start()
    count = 0
    try:
        while True:
            results = await consumer.getmany(timeout_ms=2000, max_records=50)
            if not results:
                break
            for tp_msgs in results.values():
                for msg in tp_msgs:
                    print(f"  [Run 2] partition={msg.partition}, offset={msg.offset}: {msg.value.decode()}")
                    count += 1
    finally:
        await consumer.stop()
    print(f"  [Run 2] Received {count} remaining messages")

print("=== Offset commit & resume ===")
await consume_first_half(BROKERS, TOPIC, PERSISTENT_GROUP)
print()
await consume_remainder(BROKERS, TOPIC, PERSISTENT_GROUP)

## 总结

| 场景 | 行为 |
|------|------|
| Consumer 数 = Partition 数 | 理想情况，每个 Consumer 分到 1 个 Partition |
| Consumer 数 < Partition 数 | 部分 Consumer 处理多个 Partition |
| Consumer 数 > Partition 数 | 多余 Consumer 空闲 |
| 不同 Group | 完全独立，各自消费全量数据 |
| 提交 offset | Consumer 重启后从断点续传，不重复消费 |

## 下一章

**Module 4: Replication & Fault Tolerance** — Kafka 如何通过副本保证数据不丢失，以及 Broker 宕机时 Leader 选举的过程。